In [ ]:
import kagglehub

path = kagglehub.dataset_download("ellipticco/elliptic-data-set")

print(path)

import os

print(os.listdir(path))

import torch

TORCH = torch.__version__.split("+")[0]
CUDA = "cpu"

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install -q torch-cluster -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install -q torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install -q torch-geometric

import os

class Config:
    def __init__(self):

        self.features_path = os.path.join(path, "elliptic_txs_features.csv")
        self.edges_path = os.path.join(path, "elliptic_txs_edgelist.csv")
        self.classes_path = os.path.join(path, "elliptic_txs_classes.csv")

        self.hidden_dim = 10
        self.output_dim = 3
        self.dropout = 0.6

        self.lr = 0.001
        self.weight_decay = 5e-4
        self.epochs = 150

        self.input_dim = None

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler


class EllipticDataset:
    def __init__(self, config):
        self.features_df = pd.read_csv(config.features_path, header=None)
        self.edges_df = pd.read_csv(config.edges_path)
        self.labels_df = pd.read_csv(config.classes_path)

        # Map classes
        # licit = 0, illicit = 1, unknown = 2
        self.labels_df["class"] = self.labels_df["class"].map({'unknown': 2, '1': 1, '2': 0})

        self.merged_df = self.merge()

        self.edge_index = self._edge_index()
        self.edge_weights = self._edge_weights()
        self.node_features = self._node_features()
        self.labels = self._labels()

        self.train_mask, self.val_mask, self.test_mask = self._create_masks()

    # ------------------------------------------------

    def merge(self):
        df_merge = self.features_df.merge(
            self.labels_df, how='left', right_on="txId", left_on=0
        )
        df_merge = df_merge.sort_values(0).reset_index(drop=True)
        return df_merge

    # ------------------------------------------------

    def _node_features(self):
        node_features = self.merged_df.drop(['txId'], axis=1).copy()
        node_features = node_features.drop(columns=[0, 1, "class"])

        # 🔥 Feature Normalization (VERY IMPORTANT)
        scaler = StandardScaler()
        node_features = scaler.fit_transform(node_features.values)

        return torch.tensor(node_features, dtype=torch.float)

    # ------------------------------------------------

    def _labels(self):
        labels = self.merged_df["class"].values
        return torch.tensor(labels, dtype=torch.long)

    # ------------------------------------------------

    def _edge_index(self):
        node_ids = self.merged_df[0].values
        ids_mapping = {y: x for x, y in enumerate(node_ids)}

        edges = self.edges_df.copy()
        edges.txId1 = edges.txId1.map(ids_mapping)
        edges.txId2 = edges.txId2.map(ids_mapping)
        edges = edges.astype(int)

        edge_index = torch.tensor(edges.values.T, dtype=torch.long)

        # 🔥 Make graph undirected
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

        return edge_index.contiguous()

    # ------------------------------------------------

    def _edge_weights(self):
        return torch.ones(self.edge_index.shape[1], dtype=torch.float)

    # ------------------------------------------------

    def _create_masks(self):
        labels = self.labels.numpy()
        timesteps = self.merged_df[1].values  # timestep column

        train_mask = torch.zeros(len(labels), dtype=torch.bool)
        val_mask = torch.zeros(len(labels), dtype=torch.bool)
        test_mask = torch.zeros(len(labels), dtype=torch.bool)

        # Only classified nodes (ignore unknown)
        classified = labels != 2

        # 🔥 Temporal split (proper for Elliptic)
        train_mask[(timesteps < 35) & classified] = True
        val_mask[(timesteps >= 35) & (timesteps < 40) & classified] = True
        test_mask[(timesteps >= 40) & classified] = True

        return train_mask, val_mask, test_mask

    # ------------------------------------------------

    def get_class_weights(self):
        labels_in_train = self.labels[self.train_mask]  # These will only be 0s and 1s
        class_sample_count = torch.bincount(labels_in_train) # Counts for 0 and 1

        # Determine the total number of classes. Assumes labels are 0, 1, 2.
        num_total_classes = 3 # Hardcode to 3 as per model output_dim

        # Create a weight tensor of the correct size, initialized to 0
        weights = torch.zeros(num_total_classes, dtype=torch.float)

        # Calculate inverse frequency for observed classes (0 and 1)
        for class_idx, count in enumerate(class_sample_count):
            if count > 0:
                weights[class_idx] = 1.0 / count.float()

        # Class 2 (unknown) has a weight of 0, as it's not used in training
        return weights

    # ------------------------------------------------

    def pyg_dataset(self):
        data = Data(
            x=self.node_features,
            edge_index=self.edge_index,
            edge_attr=self.edge_weights,
            y=self.labels,
            train_mask=self.train_mask,
            val_mask=self.val_mask,
            test_mask=self.test_mask
        )
        return data

!ls features.csv
!ls edgelist.csv
!ls classes.csv

import torch
import torch.nn as nn
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops


class MPNNLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(MPNNLayer, self).__init__(aggr='add')
        self.linear = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        return self.propagate(edge_index, x=x)

    def message(self, x_j):
        # Apply linear transform to neighbor features
        return self.linear(x_j)

    def update(self, aggr_out):
        return aggr_out




import torch
import torch.nn as nn
import torch.nn.functional as F


class MPNN(nn.Module):
    def __init__(self, config):
        super(MPNN, self).__init__()

        self.input_dim = config.input_dim
        self.hidden_dim = config.hidden_dim
        self.output_dim = config.output_dim
        self.dropout = config.dropout

        # Message Passing Layers
        self.mpnn1 = MPNNLayer(self.input_dim, self.hidden_dim)
        self.mpnn2 = MPNNLayer(self.hidden_dim, self.hidden_dim)

        # Final classifier (NO sigmoid)
        self.classifier = nn.Linear(self.hidden_dim, self.output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.mpnn1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.mpnn2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.classifier(x)  # raw logits

        return x


import sys
sys.path.append("/content")

!mv models.py model.py
!mv datasets.py dataset.py

import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


class MPNNTrainer:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load dataset
        self.dataset = EllipticDataset(config)
        self.data = self.dataset.pyg_dataset().to(self.device)

        # Automatically set input_dim
        self.config.input_dim = self.data.x.shape[1]

        # Model
        self.model = MPNN(config).to(self.device)

        # Class weights
        self.class_weights = self.dataset.get_class_weights().to(self.device)

        # Loss
        self.criterion = torch.nn.CrossEntropyLoss(weight=self.class_weights)

        # Optimizer
        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=config.lr,
            weight_decay=config.weight_decay
        )

    # ----------------------------
    # Training Step
    # ----------------------------
    def train_epoch(self):
        self.model.train()
        self.optimizer.zero_grad()

        out = self.model(self.data)

        loss = self.criterion(
            out[self.data.train_mask],
            self.data.y[self.data.train_mask]
        )

        loss.backward()
        self.optimizer.step()

        return loss.item()

    # ----------------------------
    # Evaluation
    # ----------------------------
    @torch.no_grad()
    def evaluate(self, mask):
        self.model.eval()
        out = self.model(self.data)

        pred = out.argmax(dim=1)

        y_true = self.data.y[mask].cpu().numpy()
        y_pred = pred[mask].cpu().numpy()

        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

        return acc, precision, recall, f1

    # ----------------------------
    # Full Training Loop
    # ----------------------------
    def run_training(self):
        best_val_recall = 0
        best_test_metrics = None

        print("Starting MPNN training...")

        for epoch in range(self.config.epochs):
            loss = self.train_epoch()

            train_metrics = self.evaluate(self.data.train_mask)
            val_metrics = self.evaluate(self.data.val_mask)
            test_metrics = self.evaluate(self.data.test_mask)

            train_acc, train_prec, train_rec, train_f1 = train_metrics
            val_acc, val_prec, val_rec, val_f1 = val_metrics
            test_acc, test_prec, test_rec, test_f1 = test_metrics

            # Select best model based on Validation Recall
            if val_rec > best_val_recall:
                best_val_recall = val_rec
                best_test_metrics = test_metrics

            if epoch % 10 == 0:
                print(f"\nEpoch {epoch:03d} | Loss: {loss:.4f}")

                print(f"Train → Acc:{train_acc:.3f} | Prec:{train_prec:.3f} | Rec:{train_rec:.3f} | F1:{train_f1:.3f}")
                print(f"Val   → Acc:{val_acc:.3f} | Prec:{val_prec:.3f} | Rec:{val_rec:.3f} | F1:{val_f1:.3f}")
                print(f"Test  → Acc:{test_acc:.3f} | Prec:{test_prec:.3f} | Rec:{test_rec:.3f} | F1:{test_f1:.3f}")

        print("\nBest Validation Recall:", best_val_recall)

        if best_test_metrics:
            best_test_acc, best_test_prec, best_test_rec, best_test_f1 = best_test_metrics
            print("Test Metrics at Best Validation Recall:")
            print(f"Test → Acc:{best_test_acc:.3f} | Prec:{best_test_prec:.3f} | Rec:{best_test_rec:.3f} | F1:{best_test_f1:.3f}")


import os

dataset_subfolder = os.path.join(path, "elliptic_bitcoin_dataset")
print(f"Contents of {dataset_subfolder}:")
print(os.listdir(dataset_subfolder))

import os

class Config:
    def __init__(self):

        # ---------------------------
        # Dataset Paths
        # ---------------------------
        dataset_subfolder = os.path.join(path, "elliptic_bitcoin_dataset")
        self.features_path = os.path.join(dataset_subfolder, "elliptic_txs_features.csv")
        self.edges_path = os.path.join(dataset_subfolder, "elliptic_txs_edgelist.csv")
        self.classes_path = os.path.join(dataset_subfolder, "elliptic_txs_classes.csv")

        # ---------------------------
        # Model Parameters
        # ---------------------------
        self.hidden_dim = 10      # Smaller than GTN (keeps MPNN weakest)
        self.output_dim = 3
        self.dropout = 0.6        # Moderate regularization

        # ---------------------------
        # Optimization
        # ---------------------------
        self.lr = 0.001
        self.weight_decay = 5e-4
        self.epochs = 150

        # Auto-filled after dataset loads
        self.input_dim = None

config = Config()

trainer = MPNNTrainer(config)

trainer.run_training()

100%|██████████| 146M/146M [00:01<00:00, 123MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/ellipticco/elliptic-data-set/versions/1
['elliptic_bitcoin_dataset']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.2/828.2 kB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.9/306.9 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.6 MB/s eta 0:00:00
ls: cannot access 'features.csv': No such file or directory
ls: cannot access 'edgelist.csv': No such file or directory
ls: cannot access 'classes.csv': No such file or directory
mv: cannot stat 'models.py': No such file or directory
mv: cannot stat 'datasets.py': No such file or directory
Contents of /root/.cache/kagglehub/datasets/ellipticco/elliptic-data-set/versions/1/elliptic_bitcoin_dataset:
['elliptic_txs_feat